# Notebook 18 — Reasoning Finetuning with GRPO

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/18_reasoning_finetuning_with_grpo.ipynb)

This notebook walks through a **small-scale** GRPO-style reasoning experiment on toy tasks with exact answers. The goal is not to pretend Colab should run giant RL jobs by default. The goal is to make rollout groups, reward shaping, logging, and length-bias trade-offs concrete on tasks you can fully inspect.

## What you will build

- a local reasoning task set with exact answers
- a toy rollout policy with several reasoning styles
- explicit reward shaping for correctness, format, and brevity
- logged training metrics across simulated GRPO updates
- small ablations for group size and length penalties

## Why this notebook stays small

Reasoning RL is easiest to understand on tasks where:

- answers are verifiable
- the action space is small enough to inspect
- reward components are transparent
- you can see when the policy starts gaming the reward

That is why we use toy arithmetic and symbolic tasks instead of opaque benchmarks.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Add notebook helpers and artifact paths

In [ ]:
from collections import Counter, defaultdict
import statistics
import re

random.seed(18)

ARTIFACT_DIR = Path("artifacts") / "notebook_18_reasoning_grpo"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def softmax_dict(weights):
    largest = max(weights.values())
    exps = {key: math.exp(value - largest) for key, value in weights.items()}
    total = sum(exps.values())
    return {key: value / total for key, value in exps.items()}

def sample_from_probs(probs):
    draw = random.random()
    cumulative = 0.0
    for key, value in probs.items():
        cumulative += value
        if draw <= cumulative:
            return key
    return next(reversed(probs))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Build a verifiable reasoning task set

We mix arithmetic, parity, and small comparison tasks. Every task has a prompt, an exact answer, and a task family so later logs can break down behavior.

In [ ]:
def build_reasoning_tasks():
    tasks = []
    for left, right in [(7, 8), (12, 19), (14, 5), (9, 9)]:
        tasks.append({
            "id": f"add_{left}_{right}",
            "family": "addition",
            "prompt": f"Compute {left} + {right}. End with FINAL: <number>.",
            "answer": str(left + right),
        })
    for value in [11, 28, 41, 64]:
        tasks.append({
            "id": f"parity_{value}",
            "family": "parity",
            "prompt": f"Is {value} odd or even? End with FINAL: odd or FINAL: even.",
            "answer": "even" if value % 2 == 0 else "odd",
        })
    for left, right in [(3, 9), (14, 6), (17, 17), (20, 4)]:
        tasks.append({
            "id": f"compare_{left}_{right}",
            "family": "comparison",
            "prompt": f"Which number is larger: {left} or {right}? End with FINAL: <number>.",
            "answer": str(max(left, right)),
        })
    return tasks

reasoning_tasks = build_reasoning_tasks()
task_df = pd.DataFrame(reasoning_tasks)
task_df.head(12)

## Step 3: Define reasoning styles for rollout samples

To keep the notebook fast, we simulate a policy over several response styles:

- concise and usually correct
- verbose and usually correct
- shortcut guessing
- sloppy formatting
- arithmetic slips

This lets us reason about GRPO dynamics without a heavy model run.

In [ ]:
STYLE_BEHAVIOR = {
    "concise_correct": {"correct_prob": 0.9, "format_prob": 0.95, "verbosity": 10},
    "verbose_correct": {"correct_prob": 0.92, "format_prob": 0.97, "verbosity": 28},
    "shortcut_guess": {"correct_prob": 0.35, "format_prob": 0.85, "verbosity": 7},
    "sloppy_format": {"correct_prob": 0.55, "format_prob": 0.25, "verbosity": 9},
    "arithmetic_slip": {"correct_prob": 0.15, "format_prob": 0.9, "verbosity": 13},
}

style_weights = {style: 0.0 for style in STYLE_BEHAVIOR}
style_weights

## Step 4: Render toy rollouts from those styles

In [ ]:
def parse_task_numbers(task):
    numbers = [int(value) for value in re.findall(r"\d+", task["prompt"])]
    return numbers

def plausible_wrong_answer(task):
    numbers = parse_task_numbers(task)
    if task["family"] == "addition":
        return str(sum(numbers) - 1)
    if task["family"] == "parity":
        return "even" if task["answer"] == "odd" else "odd"
    if task["family"] == "comparison":
        return str(min(numbers))
    return task["answer"]

def render_rollout(task, style):
    behavior = STYLE_BEHAVIOR[style]
    correct = random.random() < behavior["correct_prob"]
    formatted = random.random() < behavior["format_prob"]
    final_answer = task["answer"] if correct else plausible_wrong_answer(task)

    if style == "concise_correct":
        reasoning = f"Compute directly and keep only the needed steps. FINAL: {final_answer}"
    elif style == "verbose_correct":
        reasoning = (
            f"Step 1: read the task. Step 2: solve it carefully. "
            f"Step 3: restate the checked answer. FINAL: {final_answer}"
        )
    elif style == "shortcut_guess":
        reasoning = f"Quick guess from pattern matching. FINAL: {final_answer}"
    elif style == "sloppy_format":
        reasoning = f"I solved it and the answer is {final_answer}"
    else:
        reasoning = f"Compute with one shaky intermediate step. FINAL: {final_answer}"

    if formatted:
        text = reasoning
    else:
        text = reasoning.replace("FINAL:", "Answer")

    return {
        "task_id": task["id"],
        "family": task["family"],
        "style": style,
        "correct": correct,
        "text": text,
        "word_count": len(text.split()),
    }

demo_rollouts = [render_rollout(reasoning_tasks[0], style) for style in STYLE_BEHAVIOR]
pd.DataFrame(demo_rollouts)

## Step 5: Define reward shaping

We combine several small signals:

- correctness from exact verification
- format bonus for the required final answer tag
- reasoning bonus for useful structure
- length penalty to avoid rewarding verbosity by accident

In [ ]:
def extract_final_answer(text):
    match = re.search(r"FINAL:\s*(.+)", text.strip(), flags=re.IGNORECASE)
    return match.group(1).strip() if match else None

def score_rollout(task, rollout, length_penalty_weight=0.015):
    parsed = extract_final_answer(rollout["text"])
    correctness = 1.0 if parsed == task["answer"] else 0.0
    format_bonus = 0.2 if parsed is not None else -0.25
    reasoning_bonus = 0.1 if ("Step 1:" in rollout["text"] or "Compute" in rollout["text"]) else 0.0
    overflow = max(0, rollout["word_count"] - 18)
    brevity_penalty = -length_penalty_weight * overflow
    total = correctness + format_bonus + reasoning_bonus + brevity_penalty
    return {
        "correctness": round(correctness, 3),
        "format_bonus": round(format_bonus, 3),
        "reasoning_bonus": round(reasoning_bonus, 3),
        "brevity_penalty": round(brevity_penalty, 3),
        "total_reward": round(total, 3),
    }

sample_scores = []
for rollout in demo_rollouts:
    pieces = score_rollout(reasoning_tasks[0], rollout)
    sample_scores.append({**rollout, **pieces})

pd.DataFrame(sample_scores)[["style", "correctness", "format_bonus", "reasoning_bonus", "brevity_penalty", "total_reward", "text"]]

## Step 6: Inspect one rollout group

GRPO samples several rollouts for the same prompt, scores them, and normalizes rewards relative to the group. That makes the preference signal prompt-local instead of global.

In [ ]:
def relative_advantages(values):
    mean_value = statistics.mean(values)
    stdev = statistics.pstdev(values)
    if stdev == 0:
        return [0.0 for _ in values]
    return [round((value - mean_value) / stdev, 3) for value in values]

def sample_group(task, weights, group_size=4, length_penalty_weight=0.015):
    probs = softmax_dict(weights)
    group = []
    for _ in range(group_size):
        style = sample_from_probs(probs)
        rollout = render_rollout(task, style)
        scored = {**rollout, **score_rollout(task, rollout, length_penalty_weight=length_penalty_weight)}
        group.append(scored)
    advantages = relative_advantages([item["total_reward"] for item in group])
    for item, advantage in zip(group, advantages):
        item["advantage"] = advantage
    return group

example_group = sample_group(reasoning_tasks[1], style_weights, group_size=6)
pd.DataFrame(example_group)[["style", "correct", "total_reward", "advantage", "word_count", "text"]]

## Step 7: Run a toy GRPO training loop

The loop below is intentionally simple:

1. sample a task
2. draw a rollout group
3. compute group-relative advantages
4. nudge style weights toward above-average rollouts
5. log what happened

This is not a production trainer. It is a transparent teaching model.

In [ ]:
def train_toy_grpo(tasks, steps=90, group_size=4, lr=0.22, length_penalty_weight=0.015, seed=18):
    random.seed(seed)
    weights = {style: 0.0 for style in STYLE_BEHAVIOR}
    history = []

    for step in range(1, steps + 1):
        task = random.choice(tasks)
        group = sample_group(task, weights, group_size=group_size, length_penalty_weight=length_penalty_weight)
        for item in group:
            weights[item["style"]] += lr * item["advantage"]

        history.append({
            "step": step,
            "task_id": task["id"],
            "family": task["family"],
            "group_size": group_size,
            "mean_reward": round(statistics.mean(item["total_reward"] for item in group), 3),
            "best_reward": round(max(item["total_reward"] for item in group), 3),
            "accuracy": round(sum(item["correct"] for item in group) / len(group), 3),
            "avg_words": round(statistics.mean(item["word_count"] for item in group), 2),
            "winner_style": max(group, key=lambda item: item["total_reward"])["style"],
            "winner_advantage": max(item["advantage"] for item in group),
            "length_penalty_weight": length_penalty_weight,
        })

    return weights, pd.DataFrame(history)

final_weights, history_df = train_toy_grpo(reasoning_tasks)
history_df.head()

## Step 8: Read the logged metrics

In [ ]:
history_summary = {
    "final_mean_reward": round(history_df["mean_reward"].tail(15).mean(), 3),
    "final_accuracy": round(history_df["accuracy"].tail(15).mean(), 3),
    "final_avg_words": round(history_df["avg_words"].tail(15).mean(), 2),
}

style_probabilities = softmax_dict(final_weights)
summary_rows = [{"metric": key, "value": value} for key, value in history_summary.items()]
summary_rows.extend({"metric": f"prob_{style}", "value": round(prob, 3)} for style, prob in style_probabilities.items())

print(to_markdown_table(summary_rows, ["metric", "value"]))

## Step 9: Plot reward, accuracy, and length trends

Logged metrics matter because RL can improve one dimension while quietly harming another.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

history_df["reward_smooth"] = history_df["mean_reward"].rolling(10, min_periods=1).mean()
history_df["accuracy_smooth"] = history_df["accuracy"].rolling(10, min_periods=1).mean()
history_df["length_smooth"] = history_df["avg_words"].rolling(10, min_periods=1).mean()

axes[0].plot(history_df["step"], history_df["reward_smooth"])
axes[0].set_title("Mean reward")
axes[0].set_xlabel("step")

axes[1].plot(history_df["step"], history_df["accuracy_smooth"], color="green")
axes[1].set_title("Group accuracy")
axes[1].set_xlabel("step")

axes[2].plot(history_df["step"], history_df["length_smooth"], color="purple")
axes[2].set_title("Average output length")
axes[2].set_xlabel("step")

plt.tight_layout()
plt.show()

## Step 10: Inspect which styles are being reinforced

In [ ]:
winner_counts = history_df["winner_style"].value_counts().rename_axis("style").reset_index(name="wins")
winner_counts["share"] = (winner_counts["wins"] / winner_counts["wins"].sum()).round(3)
winner_counts

## Step 11: Length-bias notes

Longer responses often look better to a crude reward function because they contain more reasoning markers. That can create a false lesson: "be longer." We can test that by rerunning the same loop with different length penalties.

In [ ]:
def run_length_ablation(penalties):
    rows = []
    for penalty in penalties:
        weights, frame = train_toy_grpo(
            reasoning_tasks,
            steps=90,
            group_size=4,
            lr=0.22,
            length_penalty_weight=penalty,
            seed=18,
        )
        probs = softmax_dict(weights)
        rows.append({
            "length_penalty": penalty,
            "final_reward": round(frame["mean_reward"].tail(15).mean(), 3),
            "final_accuracy": round(frame["accuracy"].tail(15).mean(), 3),
            "final_avg_words": round(frame["avg_words"].tail(15).mean(), 2),
            "p_verbose_correct": round(probs["verbose_correct"], 3),
            "p_concise_correct": round(probs["concise_correct"], 3),
        })
    return pd.DataFrame(rows)

length_ablation_df = run_length_ablation([0.0, 0.01, 0.02, 0.03])
length_ablation_df

## Step 12: Plot the length-bias ablation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(length_ablation_df["length_penalty"], length_ablation_df["final_avg_words"], marker="o")
axes[0].set_title("Penalty vs average length")
axes[0].set_xlabel("length penalty")
axes[0].set_ylabel("avg words")

axes[1].plot(length_ablation_df["length_penalty"], length_ablation_df["final_accuracy"], marker="o", color="green")
axes[1].set_title("Penalty vs accuracy")
axes[1].set_xlabel("length penalty")
axes[1].set_ylabel("accuracy")

plt.tight_layout()
plt.show()

## Step 13: Vary rollout group size

Larger groups give richer relative comparisons, but they cost more samples. Small groups are cheaper but noisier.

In [ ]:
def run_group_size_ablation(group_sizes):
    rows = []
    for group_size in group_sizes:
        weights, frame = train_toy_grpo(
            reasoning_tasks,
            steps=90,
            group_size=group_size,
            lr=0.22,
            length_penalty_weight=0.015,
            seed=18,
        )
        rows.append({
            "group_size": group_size,
            "final_reward": round(frame["mean_reward"].tail(15).mean(), 3),
            "final_accuracy": round(frame["accuracy"].tail(15).mean(), 3),
            "final_avg_words": round(frame["avg_words"].tail(15).mean(), 2),
        })
    return pd.DataFrame(rows)

group_ablation_df = run_group_size_ablation([2, 4, 8])
group_ablation_df

## Step 14: Capture task-family slices

If RL only helps one family and hurts another, you want to know that before a real run ships.

In [ ]:
family_summary = (
    history_df.groupby("family")[["mean_reward", "accuracy", "avg_words"]]
    .mean()
    .round(3)
    .reset_index()
)
family_summary

## Step 15: Prepare a trainer-ready dataset and config sketch

The exact TRL or Unsloth GRPO API may shift over time. What should stay stable is the *shape* of the experiment: prompts, verifier-friendly answers, rollout group size, and logging targets.

In [ ]:
trl_ready_records = [
    {
        "prompt": task["prompt"],
        "expected_answer": task["answer"],
        "family": task["family"],
    }
    for task in reasoning_tasks
]

trl_ready_dataset = Dataset.from_list(trl_ready_records)
training_plan = {
    "algorithm": "GRPO",
    "group_size": 4,
    "reward_components": ["correctness", "format_bonus", "reasoning_bonus", "brevity_penalty"],
    "log_metrics": ["mean_reward", "accuracy", "avg_words", "winner_style"],
    "notes": "Use only on verifier-friendly tasks after SFT and evaluation baselines exist.",
}

print(trl_ready_dataset)
print(json.dumps(training_plan, indent=2))

## Step 16: Save experiment artifacts

In [ ]:
history_path = ARTIFACT_DIR / "toy_grpo_history.csv"
length_path = ARTIFACT_DIR / "length_ablation.csv"
group_path = ARTIFACT_DIR / "group_ablation.csv"
plan_path = ARTIFACT_DIR / "training_plan.json"

history_df.to_csv(history_path, index=False)
length_ablation_df.to_csv(length_path, index=False)
group_ablation_df.to_csv(group_path, index=False)
with open(plan_path, "w", encoding="utf-8") as handle:
    json.dump(training_plan, handle, indent=2)

print("Saved:", history_path)
print("Saved:", length_path)
print("Saved:", group_path)
print("Saved:", plan_path)

## Recap

You now have a compact reasoning RL workflow that makes four GRPO realities explicit:

- reward shaping is as important as the trainer choice
- rollout groups change the quality of the learning signal
- length bias can silently distort behavior
- logged metrics are mandatory because reward wins are not enough on their own